# Step 3 — Market Basket Analysis (FP-Growth)
Discovers products frequently bought together and builds a recommendation dictionary.

In [ ]:
import pandas as pd
import boto3, json, os
from mlxtend.frequent_patterns import fpgrowth, association_rules

BUCKET = 'hybrid-rec-demo-YOUR_ACCOUNT_ID'   # <-- REPLACE THIS
s3 = boto3.client('s3')
os.makedirs('../models', exist_ok=True)


In [ ]:
# Download basket matrix from S3
s3.download_file(BUCKET, 'data/processed/basket_matrix.csv', '../data/processed/basket_matrix.csv')
basket_df = pd.read_csv('../data/processed/basket_matrix.csv')
print(f'Basket matrix: {basket_df.shape[0]} transactions x {basket_df.shape[1]} products')


In [ ]:
# Run FP-Growth
frequent_itemsets = fpgrowth(basket_df, min_support=0.02, use_colnames=True)
print(f'Frequent itemsets found: {len(frequent_itemsets)}')
frequent_itemsets.sort_values('support', ascending=False).head(10)


In [ ]:
# Generate Association Rules
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
rules = rules[rules['confidence'] >= 0.30].sort_values('lift', ascending=False)
print(f'Rules generated: {len(rules)}')
rules[['antecedents','consequents','support','confidence','lift']].head(15)


In [ ]:
# Build recommendation dictionary: {product_id: [{product, score, confidence, lift}]}
rec_dict = {}
for _, row in rules.iterrows():
    for ant in row['antecedents']:
        for con in row['consequents']:
            score = float(row['lift']) * float(row['confidence'])
            if ant not in rec_dict:
                rec_dict[ant] = []
            rec_dict[ant].append({
                'product': con,
                'score': round(score, 4),
                'confidence': round(float(row['confidence']), 4),
                'lift': round(float(row['lift']), 4)
            })

for prod in rec_dict:
    rec_dict[prod] = sorted(rec_dict[prod], key=lambda x: x['score'], reverse=True)

print(f'Products with MBA recommendations: {len(rec_dict)}')

# Show example
example_prod = list(rec_dict.keys())[0]
print(f'\nSample recs for {example_prod}:')
for r in rec_dict[example_prod][:5]:
    print(f"  → {r['product']} (lift={r['lift']}, conf={r['confidence']})")


In [ ]:
# Save locally and upload to S3
with open('../models/mba_rules.json', 'w') as f:
    json.dump(rec_dict, f, indent=2)

rules.to_csv('../models/association_rules.csv', index=False)

s3.upload_file('../models/mba_rules.json', BUCKET, 'models/mba_rules.json')
s3.upload_file('../models/association_rules.csv', BUCKET, 'models/association_rules.csv')
print('MBA complete. Saved to S3.')
